# Kaggle Playground Series s6e10: TabPFN (Prior-Data Fitted Network)
**Predicting Airline Satisfaction**

In this notebook, we build an end-to-end tabular classification pipeline using **TabPFN** (*Prior-Data Fitted Networks for Tabular Data* by PriorLabs).

---

### What is TabPFN?
TabPFN is a foundation transformer model pre-trained offline on millions of synthetic tabular datasets. Unlike traditional gradient boosting trees (LightGBM, XGBoost, CatBoost) or neural networks that optimize parameters from scratch on a single dataset via iterative gradient descent:
1. **In-Context Learning**: Given a prompt consisting of training feature-label pairs, TabPFN computes posterior predictive distributions for test queries in a single forward pass.
2. **Bayesian Prior Approximation**: It acts as an approximation to Bayesian inference over causal tabular data generating processes, capturing intricate non-linear boundaries and feature interactions without hyperparameter tuning.

---

### Large Dataset Strategy (~700k Train / ~300k Test)
TabPFN is designed and optimized for context windows of up to 10,000 to 50,000 samples. Passing 700,000 samples into transformer self-attention simultaneously would exceed GPU memory limits and cause severe latency. To harness TabPFN effectively and competitively on this large-scale dataset, we implement:
1. **Stratified Bagging with 5-Fold Cross-Validation**:
   - Within each fold of 5-fold Stratified CV, we train multiple TabPFN models on diverse stratified subsets (e.g. 10,000 samples per bag).
   - Predictions are ensembled across bags and folds, combining the Bayesian prior of TabPFN with variance reduction from bagging.
2. **Chunked Memory-Safe Inference**:
   - Querying ~140k validation samples or ~300k test samples at once would trigger CUDA Out-Of-Memory (OOM). We perform inference in batched chunks (e.g. 2,500 samples), ensuring flat, minimal VRAM usage.
3. **Domain-Specific Feature Engineering**:
   - Delay composites, survey aggregations, and domain comfort/service scores consistent with competition best practices.

In [ ]:
# Install TabPFN package
# Kaggle notebooks have internet access enabled, so this installs in a few seconds.
!pip install -q tabpfn

In [ ]:
import os
import gc
import time
import random
import inspect
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.metrics import roc_auc_score, log_loss, roc_curve

import torch
from tabpfn import TabPFNClassifier

warnings.filterwarnings('ignore')

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

seed_everything(42)

# Hardware Acceleration Check: GPU is strongly recommended for TabPFN
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"PyTorch Version: {torch.__version__}")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU Model: {torch.cuda.get_device_name(0)}")
    print(f"Available VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    print("WARNING: Running on CPU. TabPFN transformer passes are dramatically faster on GPU.")

## TabPFN License Acceptance & Authentication
TabPFN requires a one-time license acceptance to download pre-trained model weights for local inference.

### How to set up your free TabPFN API Token:
1. Open [https://ux.priorlabs.ai](https://ux.priorlabs.ai) in your browser and log in (or register for free).
2. Accept the model license on the **Licenses** tab.
3. Copy your personal API Key from [https://ux.priorlabs.ai/account](https://ux.priorlabs.ai/account).
4. **On Kaggle**, you can provide this token in either of two ways:
   - **Method A (Recommended - Secure)**: In your Kaggle notebook, click **Add-ons** $\rightarrow$ **Secrets**, add a secret with label `TABPFN_TOKEN` and paste your key.
   - **Method B (Direct)**: Paste your key into the `TABPFN_TOKEN = "..."` variable in the cell below.

In [ ]:
# Authentication: Load TABPFN_TOKEN from Kaggle Secrets or direct variable
# If not using Kaggle Secrets, paste your token directly into the quotes below:
TABPFN_TOKEN = "tabpfn_sk_4c_KDLkQVN-OuExNuYGAJDXAcOo1v5DWlsCdRoqDS08"

# 1. Check Kaggle Secrets
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    token_from_secrets = secrets.get_secret("TABPFN_TOKEN")
    if token_from_secrets:
        os.environ["TABPFN_TOKEN"] = token_from_secrets
        print("✓ Successfully loaded TABPFN_TOKEN from Kaggle Secrets.")
except Exception:
    pass

# 2. Check direct variable fallback
if not os.environ.get("TABPFN_TOKEN") and TABPFN_TOKEN:
    os.environ["TABPFN_TOKEN"] = TABPFN_TOKEN.strip()
    print("✓ Successfully set TABPFN_TOKEN from notebook variable.")

# 3. Status Report
if os.environ.get("TABPFN_TOKEN"):
    masked = os.environ["TABPFN_TOKEN"][:4] + "..." + os.environ["TABPFN_TOKEN"][-4:] if len(os.environ["TABPFN_TOKEN"]) > 8 else "***"
    print(f"✓ TABPFN_TOKEN is configured: {masked}")
else:
    print("!" * 70)
    print("NOTICE: TABPFN_TOKEN is not set yet.")
    print("TabPFN requires a free API key to download weights on first use.")
    print("1. Go to https://ux.priorlabs.ai -> Accept License -> Copy API Key.")
    print("2. Add to Kaggle Secrets (Label: TABPFN_TOKEN) OR paste in TABPFN_TOKEN above.")
    print("!" * 70)

## Configuration
All hyperparameters, paths, and settings are organized in a centralized `CFG` class:
- `subsample_per_bag`: Number of training examples per TabPFN model (10,000 provides strong representation within context limits).
- `n_bags_per_fold`: Number of diverse stratified bags per fold (e.g., 2 bags × 5 folds = 10 ensemble models).
- `n_estimators`: TabPFN internal ensemble forward passes (e.g., 8 passes).
- `batch_size`: Chunk size for test/validation inference to guarantee zero GPU memory overflow.

In [ ]:
class CFG:
    competition = 'playground-series-s6e10'
    target = 'satisfaction'
    id_col = 'id'
    seed = 42
    n_splits = 5
    
    # TabPFN Ensembling & Sampling Strategy
    subsample_per_bag = 10_000   # Samples per TabPFN context (10,000 provides high representative capacity)
    n_bags_per_fold = 2          # Number of stratified bags per CV fold
    n_estimators = 8             # Number of TabPFN forward pass permutations per bag
    batch_size = 2_500           # Test / validation chunk size for memory-safe inference
    
    # Hardware device
    device = 'cuda' if torch.cuda.is_available() else 'cpu'

## Data Loading
We dynamically detect the dataset directory so that this notebook works seamlessly on Kaggle servers as well as in local environments.

In [ ]:
# Locate dataset directory
if os.path.exists(f'/kaggle/input/{CFG.competition}'):
    DATA_DIR = Path(f'/kaggle/input/{CFG.competition}')
elif os.path.exists(f'/kaggle/input/competitions/{CFG.competition}'):
    DATA_DIR = Path(f'/kaggle/input/competitions/{CFG.competition}')
else:
    DATA_DIR = Path('../../dataset')

print(f"Data Directory: {DATA_DIR}")

train_df = pd.read_csv(DATA_DIR / 'train.csv')
test_df = pd.read_csv(DATA_DIR / 'test.csv')
sample_sub = pd.read_csv(DATA_DIR / 'sample_submission.csv')

print(f"Train samples: {len(train_df):,} | Columns: {train_df.shape[1]}")
print(f"Test samples:  {len(test_df):,} | Columns: {test_df.shape[1]}")
display(train_df.head(3))

## Feature Engineering & Preprocessing
The dataset comprises:
1. **Categorical Features**: `Gender`, `Customer Type`, `Type of Travel`, `Class`.
2. **Continuous Features**: `Age`, `Flight Distance`, `Departure Delay in Minutes`, `Arrival Delay in Minutes`.
3. **Survey Features (1-5 ratings)**: 13 satisfaction survey scores.

We engineer domain features:
- **Delay Metrics**: Impute missing `Arrival Delay in Minutes` using `Departure Delay in Minutes`. Create `Total_Delay`, `Delay_Diff`, and boolean `Is_Delayed`.
- **Survey Aggregations**: Mean, sum, standard deviation, minimum, and maximum across survey ratings.
- **Domain Composite Scores**:
  - `Comfort_Score`: Mean of `Seat comfort`, `Food and drink`, `Cleanliness`, `Leg room service`.
  - `Service_Score`: Mean of `Inflight wifi service`, `Inflight entertainment`, `Online boarding`, `On-board service`, `Baggage handling`, `Checkin service`.
  - `Convenience_Score`: Mean of `Departure/Arrival time convenient`, `Ease of Online booking`, `Gate location`.
- **Categorical Encoding**: Ordinal encoding to transform categories into integer indices suitable for TabPFN.

In [ ]:
survey_cols = [
    'Inflight wifi service', 'Departure/Arrival time convenient', 'Ease of Online booking',
    'Gate location', 'Food and drink', 'Online boarding', 'Seat comfort',
    'Inflight entertainment', 'On-board service', 'Leg room service',
    'Baggage handling', 'Checkin service', 'Cleanliness'
]

cat_cols = ['Gender', 'Customer Type', 'Type of Travel', 'Class']

def engineer_features(df_train, df_test):
    df_train = df_train.copy()
    df_test = df_test.copy()
    
    for df in [df_train, df_test]:
        # Impute missing Arrival Delay with Departure Delay
        df['Arrival Delay in Minutes'] = df['Arrival Delay in Minutes'].fillna(df['Departure Delay in Minutes'])
        
        # Delay features
        df['Total_Delay'] = df['Departure Delay in Minutes'] + df['Arrival Delay in Minutes']
        df['Delay_Diff'] = df['Arrival Delay in Minutes'] - df['Departure Delay in Minutes']
        df['Is_Delayed'] = ((df['Departure Delay in Minutes'] > 0) | (df['Arrival Delay in Minutes'] > 0)).astype(float)
        
        # Survey summary statistics
        df['Rating_Mean'] = df[survey_cols].mean(axis=1)
        df['Rating_Sum'] = df[survey_cols].sum(axis=1)
        df['Rating_Std'] = df[survey_cols].std(axis=1)
        df['Rating_Min'] = df[survey_cols].min(axis=1)
        df['Rating_Max'] = df[survey_cols].max(axis=1)
        
        # Domain composite scores
        df['Comfort_Score'] = df[['Seat comfort', 'Food and drink', 'Cleanliness', 'Leg room service']].mean(axis=1)
        df['Service_Score'] = df[['Inflight wifi service', 'Inflight entertainment', 'Online boarding',
                                  'On-board service', 'Baggage handling', 'Checkin service']].mean(axis=1)
        df['Convenience_Score'] = df[['Departure/Arrival time convenient', 'Ease of Online booking', 'Gate location']].mean(axis=1)
        
    return df_train, df_test

train_eng, test_eng = engineer_features(train_df, test_df)

# Target & Test IDs
y = train_eng[CFG.target].astype(int).values
test_ids = test_eng[CFG.id_col].values

# Identify feature columns
feature_cols = [c for c in train_eng.columns if c not in [CFG.id_col, CFG.target]]
num_cols = [c for c in feature_cols if c not in cat_cols]

print(f"Total features: {len(feature_cols)}")
print(f"Categorical columns ({len(cat_cols)}): {cat_cols}")
print(f"Numerical columns ({len(num_cols)}): {num_cols}")

# Encode Categorical Columns as integers
cat_encoder = OrdinalEncoder(dtype=np.int64)
train_eng[cat_cols] = cat_encoder.fit_transform(train_eng[cat_cols])
test_eng[cat_cols] = cat_encoder.transform(test_eng[cat_cols])

# Convert feature DataFrames to float32 NumPy arrays for optimal tensor efficiency
X_train_full = train_eng[feature_cols].values.astype(np.float32)
X_test_full = test_eng[feature_cols].values.astype(np.float32)

print(f"Engineered X_train shape: {X_train_full.shape} | X_test shape: {X_test_full.shape}")

## TabPFN Model & Inference Utilities
TabPFN supports standard scikit-learn estimator methods (`fit`, `predict_proba`).

### Version-Agnostic Model Builder
To ensure seamless execution across different versions of `tabpfn` (e.g. v0.1 vs v2.x):
- We dynamically inspect `TabPFNClassifier.__init__` parameters.
- We support both `n_estimators` and `N_ensemble_configurations`.
- We enable `ignore_pretraining_limits=True` if supported.

### Memory-Safe Chunked Predictor
Inference over 140,000 validation samples or 300,000 test samples would exceed GPU memory if executed in a single call. `predict_proba_chunked` splits queries into chunks of `batch_size=2500` and extracts positive class probabilities smoothly.

In [ ]:
def build_tabpfn_classifier(device=CFG.device, n_estimators=CFG.n_estimators, random_state=CFG.seed):
    """
    Constructs a TabPFNClassifier compatible with both tabpfn v0.1.x and v2.x.
    Dynamically inspects constructor parameters to set appropriate arguments.
    """
    init_params = inspect.signature(TabPFNClassifier.__init__).parameters
    kwargs = {}
    
    if 'device' in init_params:
        kwargs['device'] = device
    if 'n_estimators' in init_params:
        kwargs['n_estimators'] = n_estimators
    elif 'N_ensemble_configurations' in init_params:
        kwargs['N_ensemble_configurations'] = n_estimators
    if 'random_state' in init_params:
        kwargs['random_state'] = random_state
    if 'ignore_pretraining_limits' in init_params:
        kwargs['ignore_pretraining_limits'] = True
        
    return TabPFNClassifier(**kwargs)

def predict_proba_chunked(model, X_query, batch_size=CFG.batch_size):
    """
    Predicts class probabilities in batched chunks to prevent GPU Out-Of-Memory errors.
    Returns 1D array of probabilities for the positive class (satisfaction=1).
    """
    n_samples = len(X_query)
    preds = []
    
    for start_idx in range(0, n_samples, batch_size):
        end_idx = min(start_idx + batch_size, n_samples)
        batch = X_query[start_idx:end_idx]
        probs = model.predict_proba(batch)
        
        # Handle multi-class output (extract positive class index 1)
        if probs.ndim == 2:
            if probs.shape[1] >= 2:
                preds.append(probs[:, 1])
            else:
                preds.append(probs[:, 0])
        else:
            preds.append(probs)
            
    return np.concatenate(preds, axis=0)

# Pre-download weights and verify license/token acceptance
print("Checking TabPFN weight download and authentication...")
try:
    test_clf = build_tabpfn_classifier(n_estimators=1)
    # Fit on 2 dummy rows to trigger weight download and check license
    test_clf.fit(np.array([[0.0, 1.0], [1.0, 0.0]], dtype=np.float32), np.array([0, 1]))
    del test_clf
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print("✓ TabPFNClassifier model weights downloaded and verified successfully!")
except Exception as e:
    print("!" * 70)
    print("ERROR / WARNING during TabPFN initialization:")
    print(e)
    print("\nIf this is a TabPFNLicenseError:")
    print("1. Visit https://ux.priorlabs.ai and accept the license.")
    print("2. Copy your API token from https://ux.priorlabs.ai/account.")
    print("3. Add it to Kaggle Secrets as 'TABPFN_TOKEN' or set TABPFN_TOKEN in Cell 4.")
    print("!" * 70)
    raise e

## Stratified 5-Fold Cross Validation with Subsampled Bagging
For each fold in the 5-fold CV:
1. Feature scaling (`StandardScaler`) is fitted strictly on the fold's training data to prevent leakage.
2. For each bag ($b = 1, \dots, \text{n\_bags\_per\_fold}$):
   - We draw a stratified random subsample of size `subsample_per_bag` ($10,000$ samples) from the fold's training set.
   - We fit `TabPFNClassifier` on this subsample.
   - We predict validation probabilities using `predict_proba_chunked`.
   - We predict test set probabilities using `predict_proba_chunked`.
3. The bagged predictions are averaged to yield the fold's Out-Of-Fold (OOF) and test predictions.
4. Memory is cleared between folds (`gc.collect()` and `torch.cuda.empty_cache()`) to maintain peak GPU efficiency.

In [ ]:
print("=" * 70)
print(f"Starting {CFG.n_splits}-Fold Cross-Validation for TabPFN")
print(f"Device: {device} | Subsample/Bag: {CFG.subsample_per_bag:,} | Bags/Fold: {CFG.n_bags_per_fold}")
print(f"TabPFN Estimators: {CFG.n_estimators} | Inference Chunk Size: {CFG.batch_size}")
print("=" * 70)

skf = StratifiedKFold(n_splits=CFG.n_splits, shuffle=True, random_state=CFG.seed)

oof_preds = np.zeros(len(train_eng), dtype=np.float32)
test_preds = np.zeros(len(test_eng), dtype=np.float32)
fold_auc_scores = []
fold_log_loss_scores = []

total_start_time = time.time()

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train_full, y)):
    print(f"\n{'=' * 25} FOLD {fold + 1} / {CFG.n_splits} {'=' * 25}")
    fold_start_time = time.time()
    
    # Train / Val splits for this fold
    X_train_fold_raw = X_train_full[train_idx]
    y_train_fold = y[train_idx]
    X_val_fold_raw = X_train_full[val_idx]
    y_val_fold = y[val_idx]
    
    # Feature Scaling: Fit scaler on training fold only to prevent data leakage
    scaler = StandardScaler()
    X_train_fold = scaler.fit_transform(X_train_fold_raw)
    X_val_fold = scaler.transform(X_val_fold_raw)
    X_test_scaled = scaler.transform(X_test_full)
    
    fold_val_bag_preds = np.zeros(len(val_idx), dtype=np.float32)
    fold_test_bag_preds = np.zeros(len(test_eng), dtype=np.float32)
    
    for bag in range(CFG.n_bags_per_fold):
        bag_seed = CFG.seed + fold * 100 + bag * 10
        print(f"  --> Bag {bag + 1}/{CFG.n_bags_per_fold} (Seed {bag_seed})...", end=" ", flush=True)
        bag_start = time.time()
        
        # Stratified sampling of training subset to maintain exact class balance
        pos_idx = np.where(y_train_fold == 1)[0]
        neg_idx = np.where(y_train_fold == 0)[0]
        pos_rate = len(pos_idx) / len(y_train_fold)
        
        n_pos_sub = int(CFG.subsample_per_bag * pos_rate)
        n_neg_sub = CFG.subsample_per_bag - n_pos_sub
        
        rng = np.random.RandomState(bag_seed)
        sub_pos = rng.choice(pos_idx, size=n_pos_sub, replace=False)
        sub_neg = rng.choice(neg_idx, size=n_neg_sub, replace=False)
        sub_idx = np.concatenate([sub_pos, sub_neg])
        rng.shuffle(sub_idx)
        
        X_sub = X_train_fold[sub_idx]
        y_sub = y_train_fold[sub_idx]
        
        # Build and fit TabPFN
        model = build_tabpfn_classifier(device=CFG.device, n_estimators=CFG.n_estimators, random_state=bag_seed)
        model.fit(X_sub, y_sub)
        
        # Predict validation probabilities in chunks
        val_probs = predict_proba_chunked(model, X_val_fold, batch_size=CFG.batch_size)
        fold_val_bag_preds += val_probs / CFG.n_bags_per_fold
        
        # Predict test probabilities in chunks
        test_probs = predict_proba_chunked(model, X_test_scaled, batch_size=CFG.batch_size)
        fold_test_bag_preds += test_probs / CFG.n_bags_per_fold
        
        bag_time = time.time() - bag_start
        bag_auc = roc_auc_score(y_val_fold, val_probs)
        print(f"Done in {bag_time:.1f}s | Bag Val AUC: {bag_auc:.5f}")
        
        # Clean up bag model
        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()
        
    # Store fold OOF predictions
    oof_preds[val_idx] = fold_val_bag_preds
    test_preds += fold_test_bag_preds / CFG.n_splits
    
    fold_auc = roc_auc_score(y_val_fold, fold_val_bag_preds)
    fold_ll = log_loss(y_val_fold, fold_val_bag_preds)
    fold_auc_scores.append(fold_auc)
    fold_log_loss_scores.append(fold_ll)
    
    fold_elapsed = time.time() - fold_start_time
    print(f"=== FOLD {fold + 1} ENSEMBLE ROC-AUC: {fold_auc:.5f} | Log Loss: {fold_ll:.5f} | Elapsed: {fold_elapsed:.1f}s ===")
    
    # Memory cleanup
    del X_train_fold, X_val_fold, X_train_fold_raw, X_val_fold_raw
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()

total_time = time.time() - total_start_time
print("\n" + "=" * 70)
print(f"Cross-Validation Completed in {total_time / 60:.2f} minutes!")
print("=" * 70)

## Out-Of-Fold Evaluation & Diagnostics
Compute overall OOF ROC-AUC and Log-Loss, print fold-by-fold summary, and plot the ROC curve along with prediction probability distributions.

In [ ]:
overall_auc = roc_auc_score(y, oof_preds)
overall_ll = log_loss(y, oof_preds)

print("-" * 50)
print(f"OVERALL 5-FOLD OOF ROC-AUC:  {overall_auc:.5f}")
print(f"OVERALL 5-FOLD OOF LOG-LOSS: {overall_ll:.5f}")
print("-" * 50)

for f, (auc_score, ll_score) in enumerate(zip(fold_auc_scores, fold_log_loss_scores)):
    print(f"  Fold {f + 1}: ROC-AUC = {auc_score:.5f} | Log-Loss = {ll_score:.5f}")

# Plotting Diagnostics: ROC Curve & Prediction Distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

# 1. ROC Curve
fpr, tpr, _ = roc_curve(y, oof_preds)
axes[0].plot(fpr, tpr, color="crimson", lw=2, label=f"TabPFN OOF (AUC = {overall_auc:.5f})")
axes[0].plot([0, 1], [0, 1], color="navy", lw=1.5, linestyle="--", label="Random Chance")
axes[0].set_xlim([0.0, 1.0])
axes[0].set_ylim([0.0, 1.05])
axes[0].set_xlabel("False Positive Rate", fontsize=11)
axes[0].set_ylabel("True Positive Rate", fontsize=11)
axes[0].set_title("Playground Series s6e10 - TabPFN ROC Curve", fontsize=12)
axes[0].legend(loc="lower right", fontsize=10)
axes[0].grid(True, alpha=0.3)

# 2. Probability Distribution by Actual Class
axes[1].hist(oof_preds[y == 0], bins=50, alpha=0.6, color="royalblue", label="Actual 0 (Dissatisfied)", density=True)
axes[1].hist(oof_preds[y == 1], bins=50, alpha=0.6, color="coral", label="Actual 1 (Satisfied)", density=True)
axes[1].set_xlabel("Predicted Probability", fontsize=11)
axes[1].set_ylabel("Density", fontsize=11)
axes[1].set_title("TabPFN OOF Prediction Distributions", fontsize=12)
axes[1].legend(loc="upper center", fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Save Out-Of-Fold predictions for blending / stacking
oof_df = pd.DataFrame({
    CFG.id_col: train_eng[CFG.id_col],
    'target': y,
    'oof_pred_prob': oof_preds
})
oof_df.to_csv('tabpfn_oof_predictions.csv', index=False)
print("Saved Out-Of-Fold predictions to tabpfn_oof_predictions.csv")

## Submission Generation
Generate the final test predictions, perform validation checks against `sample_submission.csv`, and save `submission.csv`.

In [ ]:
submission = pd.DataFrame({
    CFG.id_col: test_ids,
    CFG.target: test_preds
})

submission.to_csv('submission.csv', index=False)
print(f"Submission successfully saved to submission.csv | Total Rows: {len(submission):,}")

# Verification Checks
assert len(submission) == len(sample_sub), f"Row count mismatch: {len(submission)} vs {len(sample_sub)}"
assert list(submission.columns) == list(sample_sub.columns), f"Column mismatch: {submission.columns}"
assert not submission[CFG.target].isnull().any(), "Submission contains null/NaN values!"
assert not np.isinf(submission[CFG.target]).any(), "Submission contains infinite values!"

print("\n--- Summary Statistics of Predictions ---")
print(submission[CFG.target].describe())

print("\nFirst 10 submission rows:")
display(submission.head(10))